In [ ]:
# =========================
# CELL 1: LIBRARIES + PATHS
# =========================

import os
import gc
import json
import random
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Optional installs for Kaggle if missing
try:
    import shap
except Exception:
    !pip -q install shap
    import shap

try:
    import lime
except Exception:
    !pip -q install lime
    import lime

try:
    from imblearn.over_sampling import SMOTE
except Exception:
    !pip -q install imbalanced-learn
    from imblearn.over_sampling import SMOTE

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    auc,
    matthews_corrcoef,
    cohen_kappa_score
)
from sklearn.preprocessing import label_binarize
from sklearn.ensemble import RandomForestClassifier

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical

from lime import lime_image
from skimage.segmentation import mark_boundaries

# -------------------------
# Global settings
# -------------------------
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS_STAGE_1 = 10
EPOCHS_STAGE_2 = 8

CLAHE_CLIP_LIMIT = 1.0
USE_GAUSSIAN_BLUR = True

N_XAI = 10

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# -------------------------
# Kaggle paths
# -------------------------
INPUT_ROOT = Path("/kaggle/input/datasets/nikitamanaenkov/ultra-wide-fundus-images-for-tumor-diagnosis/data")
DATA_DIR = INPUT_ROOT / "data"

WORK_DIR = Path("/kaggle/working")
PLOT_DIR = WORK_DIR / "plots"
STATS_DIR = WORK_DIR / "stats"
MODEL_DIR = WORK_DIR / "models"
XAI_DIR = WORK_DIR / "xai_outputs"

for d in [PLOT_DIR, STATS_DIR, MODEL_DIR, XAI_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
print("Input root:", INPUT_ROOT)
print("Data dir:", DATA_DIR)
print("Working output:", WORK_DIR)

In [ ]:
# ==============================================
# CELL 2: DATASET SCAN + STATS + STRATIFIED SPLIT
# ==============================================

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

SEED = 42

# Your correct dataset path
DATA_DIR = Path("/kaggle/input/datasets/nikitamanaenkov/ultra-wide-fundus-images-for-tumor-diagnosis/data")

WORK_DIR = Path("/kaggle/working")
PLOT_DIR = WORK_DIR / "plots"
STATS_DIR = WORK_DIR / "stats"
MODEL_DIR = WORK_DIR / "models"
XAI_DIR = WORK_DIR / "xai_outputs"

for d in [PLOT_DIR, STATS_DIR, MODEL_DIR, XAI_DIR]:
    d.mkdir(parents=True, exist_ok=True)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Dataset folder not found: {DATA_DIR}")

print("Using dataset folder:")
print(DATA_DIR)

# Automatically take every subfolder as one class
class_folders = sorted([p for p in DATA_DIR.iterdir() if p.is_dir()])

print("\nDetected class folders:")
for folder in class_folders:
    print("-", folder.name)

records = []

for class_dir in class_folders:
    class_name = class_dir.name

    for img_path in class_dir.rglob("*"):
        if img_path.suffix.lower() in IMAGE_EXTS:
            records.append({
                "path": str(img_path),
                "label": class_name
            })

df = pd.DataFrame(records)

if len(df) == 0:
    raise ValueError("No images found. Check image extensions or folder structure.")

df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

class_names = sorted(df["label"].unique())
class_to_idx = {c: i for i, c in enumerate(class_names)}
idx_to_class = {i: c for c, i in class_to_idx.items()}

df["label_idx"] = df["label"].map(class_to_idx)

NUM_CLASSES = len(class_names)

print("\nTotal images:", len(df))
print("Number of classes:", NUM_CLASSES)
print("\nClasses:")
for i, c in enumerate(class_names):
    print(i, "=>", c)

print("\nSample data:")
print(df.head())

df.to_csv(STATS_DIR / "all_images_metadata.csv", index=False)

# Full dataset distribution
class_counts = df["label"].value_counts().sort_index()

stats_df = class_counts.reset_index()
stats_df.columns = ["class_name", "count"]
stats_df.to_csv(STATS_DIR / "dataset_class_distribution.csv", index=False)

plt.figure(figsize=(10, 5))
class_counts.plot(kind="bar")
plt.title("Full Dataset Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig(PLOT_DIR / "full_dataset_class_distribution.png", dpi=300)
plt.show()

# -------------------------
# 70-15-15 stratified split
# -------------------------
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED,
    stratify=df["label_idx"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label_idx"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

train_df.to_csv(STATS_DIR / "train_split.csv", index=False)
val_df.to_csv(STATS_DIR / "validation_split.csv", index=False)
test_df.to_csv(STATS_DIR / "test_split.csv", index=False)

split_stats = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "images": [len(train_df), len(val_df), len(test_df)],
    "percentage": [
        round(len(train_df) / len(df) * 100, 2),
        round(len(val_df) / len(df) * 100, 2),
        round(len(test_df) / len(df) * 100, 2)
    ]
})

split_stats.to_csv(STATS_DIR / "split_stats.csv", index=False)

print("\nSplit Stats:")
print(split_stats)

def save_split_distribution(split_name, split_df):
    counts = split_df["label"].value_counts().sort_index()

    out = counts.reset_index()
    out.columns = ["class_name", "count"]
    out.to_csv(STATS_DIR / f"{split_name}_class_distribution.csv", index=False)

    plt.figure(figsize=(10, 5))
    counts.plot(kind="bar")
    plt.title(f"{split_name.capitalize()} Class Distribution")
    plt.xlabel("Class")
    plt.ylabel("Number of Images")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.savefig(PLOT_DIR / f"{split_name}_class_distribution.png", dpi=300)
    plt.show()

save_split_distribution("train", train_df)
save_split_distribution("validation", val_df)
save_split_distribution("test", test_df)

In [ ]:
# ======================================================
# CELL 3: CLAHE 1.0 + GAUSSIAN BLUR + LOAD DATASETS RAM
# ======================================================

import os
import gc
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# Safety definitions
SEED = globals().get("SEED", 42)
IMG_SIZE = globals().get("IMG_SIZE", 224)
CLAHE_CLIP_LIMIT = globals().get("CLAHE_CLIP_LIMIT", 1.0)
USE_GAUSSIAN_BLUR = globals().get("USE_GAUSSIAN_BLUR", True)

WORK_DIR = Path("/kaggle/working")
PLOT_DIR = WORK_DIR / "plots"
STATS_DIR = WORK_DIR / "stats"
MODEL_DIR = WORK_DIR / "models"
XAI_DIR = WORK_DIR / "xai_outputs"

for d in [PLOT_DIR, STATS_DIR, MODEL_DIR, XAI_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Check previous Cell 2 variables
required_vars = ["train_df", "val_df", "test_df"]
for var in required_vars:
    if var not in globals():
        raise NameError(f"{var} is not defined. Please run Cell 2 first.")

if "class_names" not in globals():
    combined_df = pd.concat([train_df, val_df, test_df], axis=0).reset_index(drop=True)
    class_names = sorted(combined_df["label"].unique())
    class_to_idx = {c: i for i, c in enumerate(class_names)}
    idx_to_class = {i: c for c, i in class_to_idx.items()}
    NUM_CLASSES = len(class_names)

print("IMG_SIZE:", IMG_SIZE)
print("CLAHE clipLimit:", CLAHE_CLIP_LIMIT)
print("Classes:", class_names)


def crop_black_border_rgb(img_rgb, tolerance=7):
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    mask = gray > tolerance

    if mask.sum() == 0:
        return img_rgb

    coords = np.argwhere(mask)
    y0, x0 = coords.min(axis=0)
    y1, x1 = coords.max(axis=0) + 1

    return img_rgb[y0:y1, x0:x1]


def preprocess_fundus_image(path, img_size=IMG_SIZE):
    img_bgr = cv2.imread(str(path))

    if img_bgr is None:
        return None

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # Remove black ultra-wide fundus border
    img_rgb = crop_black_border_rgb(img_rgb)

    # Resize
    img_rgb = cv2.resize(
        img_rgb,
        (img_size, img_size),
        interpolation=cv2.INTER_AREA
    )

    # CLAHE 1.0 on LAB color space
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=CLAHE_CLIP_LIMIT,
        tileGridSize=(8, 8)
    )

    l_clahe = clahe.apply(l)
    lab_clahe = cv2.merge([l_clahe, a, b])
    img_clahe = cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2RGB)

    # Gaussian blur
    if USE_GAUSSIAN_BLUR:
        img_clahe = cv2.GaussianBlur(img_clahe, (3, 3), 0)

    return img_clahe.astype(np.uint8)


def load_split_to_ram(split_df, split_name):
    X_list = []
    y_list = []
    bad_files = []

    for row in tqdm(split_df.itertuples(index=False), total=len(split_df), desc=f"Loading {split_name}"):
        img = preprocess_fundus_image(row.path)

        if img is None:
            bad_files.append(row.path)
            continue

        X_list.append(img)
        y_list.append(int(row.label_idx))

    X_arr = np.stack(X_list).astype(np.uint8)
    y_arr = np.array(y_list).astype(np.int32)

    pd.DataFrame({"bad_file": bad_files}).to_csv(
        STATS_DIR / f"{split_name}_bad_files.csv",
        index=False
    )

    print(f"\n{split_name} X shape:", X_arr.shape)
    print(f"{split_name} y shape:", y_arr.shape)
    print(f"{split_name} RAM usage:", round(X_arr.nbytes / (1024 ** 3), 3), "GB")
    print(f"{split_name} bad files:", len(bad_files))

    return X_arr, y_arr


# Load train, validation, test into RAM
X_train, y_train = load_split_to_ram(train_df, "train")
X_val, y_val = load_split_to_ram(val_df, "validation")
X_test, y_test = load_split_to_ram(test_df, "test")

# Save RAM arrays if needed later
np.save(STATS_DIR / "y_train.npy", y_train)
np.save(STATS_DIR / "y_val.npy", y_val)
np.save(STATS_DIR / "y_test.npy", y_test)

# Show preprocessing examples
sample_df = pd.concat([train_df, val_df, test_df], axis=0).groupby("label").head(1).reset_index(drop=True)

plt.figure(figsize=(12, len(sample_df) * 3))

for i, row in sample_df.iterrows():
    raw_bgr = cv2.imread(row["path"])
    raw_rgb = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)
    proc = preprocess_fundus_image(row["path"])

    plt.subplot(len(sample_df), 2, 2 * i + 1)
    plt.imshow(raw_rgb)
    plt.title(f"Original: {row['label']}")
    plt.axis("off")

    plt.subplot(len(sample_df), 2, 2 * i + 2)
    plt.imshow(proc)
    plt.title("CLAHE 1.0 + Gaussian Blur")
    plt.axis("off")

plt.tight_layout()
plt.savefig(PLOT_DIR / "clahe_preprocessing_examples.png", dpi=300)
plt.show()

gc.collect()

In [ ]:
# CELL 4

import gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from imblearn.over_sampling import SMOTE
except Exception:
    !pip -q install imbalanced-learn
    from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical

SEED = globals().get("SEED", 42)
IMG_SIZE = globals().get("IMG_SIZE", 224)
BATCH_SIZE = globals().get("BATCH_SIZE", 16)

WORK_DIR = Path("/kaggle/working")
PLOT_DIR = WORK_DIR / "plots"
STATS_DIR = WORK_DIR / "stats"
MODEL_DIR = WORK_DIR / "models"
XAI_DIR = WORK_DIR / "xai_outputs"

for d in [PLOT_DIR, STATS_DIR, MODEL_DIR, XAI_DIR]:
    d.mkdir(parents=True, exist_ok=True)

required_vars = [
    "X_train", "y_train",
    "X_val", "y_val",
    "X_test", "y_test",
    "class_names"
]

for var in required_vars:
    if var not in globals():
        raise NameError(f"{var} is not defined. Run Cells 1, 2, and 3 first.")

NUM_CLASSES = globals().get("NUM_CLASSES", len(class_names))
idx_to_class = globals().get(
    "idx_to_class",
    {i: c for i, c in enumerate(class_names)}
)

print("Before SMOTE train shape:", X_train.shape)
print("Before SMOTE distribution:", np.bincount(y_train, minlength=NUM_CLASSES))

class_counts = np.bincount(y_train, minlength=NUM_CLASSES)
min_class_count = class_counts.min()

if min_class_count < 2:
    raise ValueError(
        "SMOTE cannot run because at least one class has fewer than 2 samples in train set."
    )

k_neighbors = max(1, min(3, min_class_count - 1))

print("SMOTE k_neighbors:", k_neighbors)

X_train_flat = X_train.reshape((X_train.shape[0], -1)).astype(np.float32)

smote = SMOTE(
    random_state=SEED,
    k_neighbors=k_neighbors
)

X_train_smote_flat, y_train_smote = smote.fit_resample(
    X_train_flat,
    y_train
)

X_train_smote = X_train_smote_flat.reshape(
    (-1, IMG_SIZE, IMG_SIZE, 3)
)

X_train_smote = np.clip(X_train_smote, 0, 255).astype(np.uint8)
y_train_smote = y_train_smote.astype(np.int32)

del X_train_flat, X_train_smote_flat
gc.collect()

print("After SMOTE train shape:", X_train_smote.shape)
print("After SMOTE distribution:", np.bincount(y_train_smote, minlength=NUM_CLASSES))

smote_stats = pd.DataFrame({
    "class_name": [idx_to_class[i] for i in range(NUM_CLASSES)],
    "before_smote": np.bincount(y_train, minlength=NUM_CLASSES),
    "after_smote": np.bincount(y_train_smote, minlength=NUM_CLASSES)
})

smote_stats.to_csv(
    STATS_DIR / "smote_train_only_distribution.csv",
    index=False
)

print(smote_stats)

plt.figure(figsize=(10, 5))

x = np.arange(NUM_CLASSES)
width = 0.35

plt.bar(
    x - width / 2,
    smote_stats["before_smote"],
    width,
    label="Before SMOTE"
)

plt.bar(
    x + width / 2,
    smote_stats["after_smote"],
    width,
    label="After SMOTE"
)

plt.xticks(x, class_names, rotation=35, ha="right")
plt.title("Train Set Distribution Before and After SMOTE")
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / "smote_train_distribution.png", dpi=300)
plt.show()

y_train_cat = to_categorical(y_train_smote, NUM_CLASSES)
y_val_cat = to_categorical(y_val, NUM_CLASSES)
y_test_cat = to_categorical(y_test, NUM_CLASSES)

train_datagen = ImageDataGenerator(
    rotation_range=8,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.10,
    horizontal_flip=True,
    fill_mode="nearest"
)

eval_datagen = ImageDataGenerator()

train_gen = train_datagen.flow(
    X_train_smote,
    y_train_cat,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

val_gen = eval_datagen.flow(
    X_val,
    y_val_cat,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_gen = eval_datagen.flow(
    X_test,
    y_test_cat,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Train generator batches:", len(train_gen))
print("Validation generator batches:", len(val_gen))
print("Test generator batches:", len(test_gen))
print("Cell 4 completed successfully.")

In [ ]:
# CELL 5

from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = globals().get("SEED", 42)
IMG_SIZE = globals().get("IMG_SIZE", 224)
NUM_CLASSES = globals().get(
    "NUM_CLASSES",
    len(class_names) if "class_names" in globals() else 6
)

WORK_DIR = Path("/kaggle/working")
PLOT_DIR = WORK_DIR / "plots"
STATS_DIR = WORK_DIR / "stats"
MODEL_DIR = WORK_DIR / "models"
XAI_DIR = WORK_DIR / "xai_outputs"

for d in [PLOT_DIR, STATS_DIR, MODEL_DIR, XAI_DIR]:
    d.mkdir(parents=True, exist_ok=True)

tf.random.set_seed(SEED)

required_vars = [
    "train_gen",
    "val_gen",
    "test_gen",
    "class_names"
]

for var in required_vars:
    if var not in globals():
        raise NameError(f"{var} is not defined. Run Cells 1 to 4 first.")

print("IMG_SIZE:", IMG_SIZE)
print("NUM_CLASSES:", NUM_CLASSES)
print("Classes:", class_names)


def sanitize_name(name):
    return (
        name.replace("/", "_")
        .replace(" ", "_")
        .replace("-", "_")
        .replace("(", "")
        .replace(")", "")
        .replace(".", "_")
    )


def get_backbone_app(backbone_name):
    apps = {
        "EfficientNetB0": keras.applications.EfficientNetB0,
        "MobileNetV2": keras.applications.MobileNetV2,
        "ResNet50": keras.applications.ResNet50,
        "DenseNet121": keras.applications.DenseNet121,
        "VGG16": keras.applications.VGG16,
        "VGG19": keras.applications.VGG19,
        "NASNetMobile": keras.applications.NASNetMobile,
        "InceptionV3": keras.applications.InceptionV3,
        "Xception": keras.applications.Xception,
        "InceptionResNetV2": keras.applications.InceptionResNetV2,
    }

    if backbone_name not in apps:
        raise ValueError(f"Unknown backbone: {backbone_name}")

    return apps[backbone_name]


def get_preprocess_fn(backbone_name):
    preprocess_fns = {
        "EfficientNetB0": keras.applications.efficientnet.preprocess_input,
        "MobileNetV2": keras.applications.mobilenet_v2.preprocess_input,
        "ResNet50": keras.applications.resnet.preprocess_input,
        "DenseNet121": keras.applications.densenet.preprocess_input,
        "VGG16": keras.applications.vgg16.preprocess_input,
        "VGG19": keras.applications.vgg19.preprocess_input,
        "NASNetMobile": keras.applications.nasnet.preprocess_input,
        "InceptionV3": keras.applications.inception_v3.preprocess_input,
        "Xception": keras.applications.xception.preprocess_input,
        "InceptionResNetV2": keras.applications.inception_resnet_v2.preprocess_input,
    }

    if backbone_name not in preprocess_fns:
        raise ValueError(f"No preprocess function found for: {backbone_name}")

    return preprocess_fns[backbone_name]


def se_block(x, ratio=16, name="se"):
    channels = int(x.shape[-1])

    se = layers.GlobalAveragePooling2D(
        name=f"{name}_gap"
    )(x)

    se = layers.Dense(
        max(channels // ratio, 8),
        activation="relu",
        name=f"{name}_fc1"
    )(se)

    se = layers.Dense(
        channels,
        activation="sigmoid",
        name=f"{name}_fc2"
    )(se)

    se = layers.Reshape(
        (1, 1, channels),
        name=f"{name}_reshape"
    )(se)

    x = layers.Multiply(
        name=f"{name}_scale"
    )([x, se])

    return x


def cbam_block(x, ratio=16, name="cbam"):
    channels = int(x.shape[-1])

    avg_pool = layers.GlobalAveragePooling2D(
        name=f"{name}_avg_pool"
    )(x)

    max_pool = layers.GlobalMaxPooling2D(
        name=f"{name}_max_pool"
    )(x)

    shared_dense_1 = layers.Dense(
        max(channels // ratio, 8),
        activation="relu",
        name=f"{name}_shared_fc1"
    )

    shared_dense_2 = layers.Dense(
        channels,
        activation=None,
        name=f"{name}_shared_fc2"
    )

    avg_out = shared_dense_2(shared_dense_1(avg_pool))
    max_out = shared_dense_2(shared_dense_1(max_pool))

    channel_attention = layers.Add(
        name=f"{name}_channel_add"
    )([avg_out, max_out])

    channel_attention = layers.Activation(
        "sigmoid",
        name=f"{name}_channel_sigmoid"
    )(channel_attention)

    channel_attention = layers.Reshape(
        (1, 1, channels),
        name=f"{name}_channel_reshape"
    )(channel_attention)

    x = layers.Multiply(
        name=f"{name}_channel_scale"
    )([x, channel_attention])

    avg_spatial = layers.Lambda(
        lambda z: tf.reduce_mean(z, axis=-1, keepdims=True),
        name=f"{name}_spatial_avg"
    )(x)

    max_spatial = layers.Lambda(
        lambda z: tf.reduce_max(z, axis=-1, keepdims=True),
        name=f"{name}_spatial_max"
    )(x)

    spatial_attention = layers.Concatenate(
        axis=-1,
        name=f"{name}_spatial_concat"
    )([avg_spatial, max_spatial])

    spatial_attention = layers.Conv2D(
        filters=1,
        kernel_size=7,
        padding="same",
        activation="sigmoid",
        name=f"{name}_spatial_conv"
    )(spatial_attention)

    x = layers.Multiply(
        name=f"{name}_spatial_scale"
    )([x, spatial_attention])

    return x


def aspp_block(x, filters=128, name="aspp"):
    b1 = layers.Conv2D(
        filters,
        kernel_size=1,
        padding="same",
        activation="swish",
        name=f"{name}_conv_1x1"
    )(x)

    b2 = layers.Conv2D(
        filters,
        kernel_size=3,
        dilation_rate=2,
        padding="same",
        activation="swish",
        name=f"{name}_conv_d2"
    )(x)

    b3 = layers.Conv2D(
        filters,
        kernel_size=3,
        dilation_rate=4,
        padding="same",
        activation="swish",
        name=f"{name}_conv_d4"
    )(x)

    b4 = layers.Conv2D(
        filters,
        kernel_size=3,
        dilation_rate=6,
        padding="same",
        activation="swish",
        name=f"{name}_conv_d6"
    )(x)

    x = layers.Concatenate(
        name=f"{name}_concat"
    )([b1, b2, b3, b4])

    x = layers.Conv2D(
        filters * 2,
        kernel_size=1,
        padding="same",
        activation="swish",
        name=f"{name}_fusion"
    )(x)

    return x


def apply_hybrid_block(x, hybrid_type, model_key):
    if hybrid_type is None:
        return x

    if hybrid_type == "SE":
        return se_block(
            x,
            name=f"{model_key}_se"
        )

    if hybrid_type == "CBAM":
        return cbam_block(
            x,
            name=f"{model_key}_cbam"
        )

    if hybrid_type == "ASPP":
        return aspp_block(
            x,
            filters=128,
            name=f"{model_key}_aspp"
        )

    raise ValueError(f"Unknown hybrid block: {hybrid_type}")


def build_model_from_config(config, img_size=IMG_SIZE, num_classes=NUM_CLASSES):
    model_name = config["model_name"]
    backbone_name = config["backbone"]
    hybrid_type = config.get("hybrid", None)
    use_imagenet = config.get("imagenet", True)

    model_key = sanitize_name(model_name)

    inputs = keras.Input(
        shape=(img_size, img_size, 3),
        name=f"{model_key}_input"
    )

    preprocess_fn = get_preprocess_fn(backbone_name)

    x = layers.Lambda(
        lambda z: preprocess_fn(z),
        name=f"{model_key}_preprocess"
    )(inputs)

    backbone_app = get_backbone_app(backbone_name)

    try:
        base_model = backbone_app(
            include_top=False,
            weights="imagenet" if use_imagenet else None,
            input_tensor=x
        )

        print(f"{model_name}: ImageNet weights loaded.")

    except Exception as e:
        print(f"{model_name}: ImageNet loading failed. Using random weights.")
        print("Reason:", e)

        base_model = backbone_app(
            include_top=False,
            weights=None,
            input_tensor=x
        )

    base_model.trainable = False

    x = base_model.output

    x = apply_hybrid_block(
        x=x,
        hybrid_type=hybrid_type,
        model_key=model_key
    )

    x = layers.Conv2D(
        filters=256,
        kernel_size=1,
        padding="same",
        activation="swish",
        name="xai_conv"
    )(x)

    x = layers.BatchNormalization(
        name="xai_bn"
    )(x)

    x = layers.GlobalAveragePooling2D(
        name="embedding"
    )(x)

    x = layers.Dropout(
        rate=0.40,
        name="dropout"
    )(x)

    outputs = layers.Dense(
        units=num_classes,
        activation="softmax",
        name="predictions"
    )(x)

    model = keras.Model(
        inputs=inputs,
        outputs=outputs,
        name=model_key
    )

    return model, base_model


simple_imagenet_models = [
    {
        "model_name": "EfficientNetB0",
        "backbone": "EfficientNetB0",
        "hybrid": None,
        "type": "Simple_ImageNet",
        "imagenet": True
    },
    {
        "model_name": "MobileNetV2",
        "backbone": "MobileNetV2",
        "hybrid": None,
        "type": "Simple_ImageNet",
        "imagenet": True
    },
    {
        "model_name": "ResNet50",
        "backbone": "ResNet50",
        "hybrid": None,
        "type": "Simple_ImageNet",
        "imagenet": True
    },
    {
        "model_name": "DenseNet121",
        "backbone": "DenseNet121",
        "hybrid": None,
        "type": "Simple_ImageNet",
        "imagenet": True
    },
    {
        "model_name": "VGG16",
        "backbone": "VGG16",
        "hybrid": None,
        "type": "Simple_ImageNet",
        "imagenet": True
    },
    {
        "model_name": "VGG19",
        "backbone": "VGG19",
        "hybrid": None,
        "type": "Simple_ImageNet",
        "imagenet": True
    },
    {
        "model_name": "NASNetMobile",
        "backbone": "NASNetMobile",
        "hybrid": None,
        "type": "Simple_ImageNet",
        "imagenet": True
    },
    {
        "model_name": "InceptionV3",
        "backbone": "InceptionV3",
        "hybrid": None,
        "type": "Simple_ImageNet",
        "imagenet": True
    },
    {
        "model_name": "Xception",
        "backbone": "Xception",
        "hybrid": None,
        "type": "Simple_ImageNet",
        "imagenet": True
    },
    {
        "model_name": "InceptionResNetV2",
        "backbone": "InceptionResNetV2",
        "hybrid": None,
        "type": "Simple_ImageNet",
        "imagenet": True
    }
]


hybrid_backbones = [
    "EfficientNetB0",
    "MobileNetV2",
    "ResNet50",
    "DenseNet121",
    "NASNetMobile"
]

hybrid_blocks = [
    "SE",
    "CBAM",
    "ASPP"
]

hybrid_models = []

for backbone_name in hybrid_backbones:
    for hybrid_block in hybrid_blocks:
        hybrid_models.append({
            "model_name": f"{backbone_name}_{hybrid_block}",
            "backbone": backbone_name,
            "hybrid": hybrid_block,
            "type": "Hybrid_ImageNet",
            "imagenet": True
        })

MODEL_CONFIGS = simple_imagenet_models + hybrid_models

model_config_df = pd.DataFrame(MODEL_CONFIGS)

model_config_df.to_csv(
    STATS_DIR / "model_configurations_25_models.csv",
    index=False
)

print("Total models:", len(MODEL_CONFIGS))
print("Simple ImageNet models:", len(simple_imagenet_models))
print("Hybrid models:", len(hybrid_models))
print(model_config_df)

print("Cell 5 completed successfully.")

In [ ]:
from pathlib import Path
import gc
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras

SEED = globals().get("SEED", 42)
IMG_SIZE = globals().get("IMG_SIZE", 224)
NUM_CLASSES = globals().get(
    "NUM_CLASSES",
    len(class_names) if "class_names" in globals() else 6
)

EPOCHS_STAGE_1 = globals().get("EPOCHS_STAGE_1", 10)

WORK_DIR = Path("/kaggle/working")
PLOT_DIR = WORK_DIR / "plots"
STATS_DIR = WORK_DIR / "stats"
MODEL_DIR = WORK_DIR / "models"
XAI_DIR = WORK_DIR / "xai_outputs"

for d in [PLOT_DIR, STATS_DIR, MODEL_DIR, XAI_DIR]:
    d.mkdir(parents=True, exist_ok=True)

required_vars = [
    "MODEL_CONFIGS",
    "build_model_from_config",
    "sanitize_name",
    "train_gen",
    "val_gen",
    "class_names"
]

for var in required_vars:
    if var not in globals():
        raise NameError(f"{var} is not defined. Run Cells 1 to 5 first.")

ALL_MODELS_DIR = MODEL_DIR / "all_25_model_runs_stage1_only"
ALL_MODELS_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_DIR = MODEL_DIR / "best_model_stage1_only"
BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)

comparison_results = []
all_histories = {}

BEST_VAL_ACCURACY = -1.0
BEST_MODEL_NAME = None
BEST_MODEL_KEY = None
BEST_MODEL_SOURCE_WEIGHTS = None

BEST_MODEL_SAVED_PATH = BEST_MODEL_DIR / "best_overall_model_stage1.keras"
BEST_WEIGHTS_SAVED_PATH = BEST_MODEL_DIR / "best_overall_weights_stage1.weights.h5"

for model_number, config in enumerate(MODEL_CONFIGS, start=1):
    model_name = config["model_name"]
    model_key = sanitize_name(model_name)

    print("\n" + "=" * 100)
    print(f"MODEL {model_number}/{len(MODEL_CONFIGS)}: {model_name}")
    print("TRAINING MODE: STAGE 1 ONLY - frozen backbone")
    print("=" * 100)

    model_run_dir = ALL_MODELS_DIR / model_key
    model_run_dir.mkdir(parents=True, exist_ok=True)

    keras.backend.clear_session()
    gc.collect()
    tf.random.set_seed(SEED)

    try:
        model, base_model = build_model_from_config(
            config=config,
            img_size=IMG_SIZE,
            num_classes=NUM_CLASSES
        )

        base_model.trainable = False

        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=1e-3),
            loss="categorical_crossentropy",
            metrics=[
                "accuracy",
                keras.metrics.AUC(name="auc", multi_label=True)
            ]
        )

        summary_path = model_run_dir / f"{model_key}_summary.txt"

        with open(summary_path, "w") as f:
            model.summary(print_fn=lambda x: f.write(x + "\n"))

        best_weights_path = model_run_dir / f"{model_key}_best_weights_stage1.weights.h5"
        final_weights_path = model_run_dir / f"{model_key}_final_weights_stage1.weights.h5"
        final_model_path = model_run_dir / f"{model_key}_final_model_stage1.keras"
        history_path = model_run_dir / f"{model_key}_history_stage1.csv"
        training_log_path = model_run_dir / f"{model_key}_training_log_stage1.csv"

        callbacks = [
            keras.callbacks.ModelCheckpoint(
                filepath=str(best_weights_path),
                monitor="val_accuracy",
                mode="max",
                save_best_only=True,
                save_weights_only=True,
                verbose=1
            ),
            keras.callbacks.EarlyStopping(
                monitor="val_accuracy",
                mode="max",
                patience=5,
                restore_best_weights=True,
                verbose=1
            ),
            keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss",
                factor=0.3,
                patience=2,
                min_lr=1e-7,
                verbose=1
            ),
            keras.callbacks.CSVLogger(
                filename=str(training_log_path),
                append=False
            )
        ]

        history_1 = model.fit(
            train_gen,
            validation_data=val_gen,
            epochs=EPOCHS_STAGE_1,
            callbacks=callbacks,
            verbose=1
        )

        history_df = pd.DataFrame(history_1.history)
        history_df.to_csv(history_path, index=False)

        all_histories[model_name] = history_df

        if best_weights_path.exists():
            model.load_weights(str(best_weights_path))

        model.save(final_model_path)
        model.save_weights(final_weights_path)

        best_epoch = int(np.argmax(history_df["val_accuracy"]) + 1)

        best_train_accuracy = float(np.max(history_df["accuracy"]))
        best_val_accuracy = float(np.max(history_df["val_accuracy"]))
        final_train_accuracy = float(history_df["accuracy"].iloc[-1])
        final_val_accuracy = float(history_df["val_accuracy"].iloc[-1])

        best_train_loss = float(np.min(history_df["loss"]))
        best_val_loss = float(np.min(history_df["val_loss"]))
        final_train_loss = float(history_df["loss"].iloc[-1])
        final_val_loss = float(history_df["val_loss"].iloc[-1])

        best_train_auc = float(np.max(history_df["auc"])) if "auc" in history_df.columns else np.nan
        best_val_auc = float(np.max(history_df["val_auc"])) if "val_auc" in history_df.columns else np.nan
        final_train_auc = float(history_df["auc"].iloc[-1]) if "auc" in history_df.columns else np.nan
        final_val_auc = float(history_df["val_auc"].iloc[-1]) if "val_auc" in history_df.columns else np.nan

        comparison_results.append({
            "model_number": model_number,
            "model_name": model_name,
            "model_key": model_key,
            "model_type": config["type"],
            "backbone": config["backbone"],
            "hybrid_block": config.get("hybrid", None),
            "training_mode": "stage1_only_frozen_backbone",
            "best_epoch": best_epoch,
            "best_train_accuracy": best_train_accuracy,
            "best_val_accuracy": best_val_accuracy,
            "final_train_accuracy": final_train_accuracy,
            "final_val_accuracy": final_val_accuracy,
            "best_train_loss": best_train_loss,
            "best_val_loss": best_val_loss,
            "final_train_loss": final_train_loss,
            "final_val_loss": final_val_loss,
            "best_train_auc": best_train_auc,
            "best_val_auc": best_val_auc,
            "final_train_auc": final_train_auc,
            "final_val_auc": final_val_auc,
            "best_weights_path": str(best_weights_path),
            "final_weights_path": str(final_weights_path),
            "final_model_path": str(final_model_path),
            "history_path": str(history_path),
            "summary_path": str(summary_path),
            "training_status": "completed",
            "error": ""
        })

        plt.figure(figsize=(10, 5))
        plt.plot(history_df["accuracy"], label="Train Accuracy")
        plt.plot(history_df["val_accuracy"], label="Validation Accuracy")
        plt.title(f"{model_name} Accuracy Curve - Stage 1 Only")
        plt.xlabel("Epoch")
        plt.ylabel("Accuracy")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(model_run_dir / f"{model_key}_accuracy_curve_stage1.png", dpi=300)
        plt.show()
        plt.close()

        plt.figure(figsize=(10, 5))
        plt.plot(history_df["loss"], label="Train Loss")
        plt.plot(history_df["val_loss"], label="Validation Loss")
        plt.title(f"{model_name} Loss Curve - Stage 1 Only")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(model_run_dir / f"{model_key}_loss_curve_stage1.png", dpi=300)
        plt.show()
        plt.close()

        if "auc" in history_df.columns and "val_auc" in history_df.columns:
            plt.figure(figsize=(10, 5))
            plt.plot(history_df["auc"], label="Train AUC")
            plt.plot(history_df["val_auc"], label="Validation AUC")
            plt.title(f"{model_name} AUC Curve - Stage 1 Only")
            plt.xlabel("Epoch")
            plt.ylabel("AUC")
            plt.legend()
            plt.grid(True)
            plt.tight_layout()
            plt.savefig(model_run_dir / f"{model_key}_auc_curve_stage1.png", dpi=300)
            plt.show()
            plt.close()

        if best_val_accuracy > BEST_VAL_ACCURACY:
            BEST_VAL_ACCURACY = best_val_accuracy
            BEST_MODEL_NAME = model_name
            BEST_MODEL_KEY = model_key
            BEST_MODEL_SOURCE_WEIGHTS = best_weights_path

            if best_weights_path.exists():
                model.load_weights(str(best_weights_path))

            model.save(BEST_MODEL_SAVED_PATH)
            model.save_weights(BEST_WEIGHTS_SAVED_PATH)

            best_info = {
                "best_model_name": BEST_MODEL_NAME,
                "best_model_key": BEST_MODEL_KEY,
                "best_validation_accuracy": BEST_VAL_ACCURACY,
                "training_mode": "stage1_only_frozen_backbone",
                "best_source_weights": str(BEST_MODEL_SOURCE_WEIGHTS),
                "best_model_saved_path": str(BEST_MODEL_SAVED_PATH),
                "best_weights_saved_path": str(BEST_WEIGHTS_SAVED_PATH)
            }

            with open(BEST_MODEL_DIR / "best_model_info.json", "w") as f:
                json.dump(best_info, f, indent=4)

            print("\nNEW BEST MODEL SAVED")
            print(json.dumps(best_info, indent=4))

        live_df = pd.DataFrame(comparison_results)
        live_df.to_csv(STATS_DIR / "model_comparison_25_models_stage1_live.csv", index=False)

        del model
        del base_model
        gc.collect()
        keras.backend.clear_session()

    except Exception as e:
        print(f"{model_name} failed:", e)

        comparison_results.append({
            "model_number": model_number,
            "model_name": model_name,
            "model_key": model_key,
            "model_type": config.get("type", ""),
            "backbone": config.get("backbone", ""),
            "hybrid_block": config.get("hybrid", None),
            "training_mode": "stage1_only_frozen_backbone",
            "best_epoch": np.nan,
            "best_train_accuracy": np.nan,
            "best_val_accuracy": np.nan,
            "final_train_accuracy": np.nan,
            "final_val_accuracy": np.nan,
            "best_train_loss": np.nan,
            "best_val_loss": np.nan,
            "final_train_loss": np.nan,
            "final_val_loss": np.nan,
            "best_train_auc": np.nan,
            "best_val_auc": np.nan,
            "final_train_auc": np.nan,
            "final_val_auc": np.nan,
            "best_weights_path": "",
            "final_weights_path": "",
            "final_model_path": "",
            "history_path": "",
            "summary_path": "",
            "training_status": "failed",
            "error": str(e)
        })

        live_df = pd.DataFrame(comparison_results)
        live_df.to_csv(STATS_DIR / "model_comparison_25_models_stage1_live.csv", index=False)

        gc.collect()
        keras.backend.clear_session()

comparison_df = pd.DataFrame(comparison_results)

comparison_df = comparison_df.sort_values(
    by="best_val_accuracy",
    ascending=False,
    na_position="last"
).reset_index(drop=True)

comparison_df["final_rank"] = np.arange(1, len(comparison_df) + 1)

comparison_csv_path = STATS_DIR / "model_comparison_25_models_stage1_only.csv"

comparison_df.to_csv(
    comparison_csv_path,
    index=False
)

print("\n" + "=" * 100)
print("FINAL 25 MODEL COMPARISON TABLE - STAGE 1 ONLY")
print("=" * 100)
print(comparison_df)

completed_df = comparison_df[
    comparison_df["training_status"] == "completed"
].copy()

if len(completed_df) > 0:
    plt.figure(figsize=(18, 7))
    plt.bar(
        completed_df["model_name"],
        completed_df["best_val_accuracy"]
    )
    plt.title("25 DL Models Comparison - Best Validation Accuracy - Stage 1 Only")
    plt.xlabel("Model")
    plt.ylabel("Best Validation Accuracy")
    plt.xticks(rotation=75, ha="right")
    plt.grid(axis="y")
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "25_models_best_validation_accuracy_stage1_only.png", dpi=300)
    plt.show()
    plt.close()

    plt.figure(figsize=(18, 7))
    plt.bar(
        completed_df["model_name"],
        completed_df["best_val_auc"]
    )
    plt.title("25 DL Models Comparison - Best Validation AUC - Stage 1 Only")
    plt.xlabel("Model")
    plt.ylabel("Best Validation AUC")
    plt.xticks(rotation=75, ha="right")
    plt.grid(axis="y")
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "25_models_best_validation_auc_stage1_only.png", dpi=300)
    plt.show()
    plt.close()

    plt.figure(figsize=(18, 7))
    plt.bar(
        completed_df["model_name"],
        completed_df["best_val_loss"]
    )
    plt.title("25 DL Models Comparison - Best Validation Loss - Stage 1 Only")
    plt.xlabel("Model")
    plt.ylabel("Best Validation Loss")
    plt.xticks(rotation=75, ha="right")
    plt.grid(axis="y")
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "25_models_best_validation_loss_stage1_only.png", dpi=300)
    plt.show()
    plt.close()

    top_10_df = completed_df.head(10)

    plt.figure(figsize=(14, 6))
    plt.bar(
        top_10_df["model_name"],
        top_10_df["best_val_accuracy"]
    )
    plt.title("Top 10 Models by Validation Accuracy - Stage 1 Only")
    plt.xlabel("Model")
    plt.ylabel("Best Validation Accuracy")
    plt.xticks(rotation=45, ha="right")
    plt.grid(axis="y")
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "top_10_models_validation_accuracy_stage1_only.png", dpi=300)
    plt.show()
    plt.close()

    plt.figure(figsize=(16, 8))
    for model_name, history_df in all_histories.items():
        if "val_accuracy" in history_df.columns:
            plt.plot(
                history_df["val_accuracy"].values,
                label=model_name,
                linewidth=1.5
            )

    plt.title("Validation Accuracy Curves - All Completed Models - Stage 1 Only")
    plt.xlabel("Epoch")
    plt.ylabel("Validation Accuracy")
    plt.legend(fontsize=7, ncol=2)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "all_models_validation_accuracy_curves_stage1_only.png", dpi=300)
    plt.show()
    plt.close()

    plt.figure(figsize=(16, 8))
    for model_name, history_df in all_histories.items():
        if "val_loss" in history_df.columns:
            plt.plot(
                history_df["val_loss"].values,
                label=model_name,
                linewidth=1.5
            )

    plt.title("Validation Loss Curves - All Completed Models - Stage 1 Only")
    plt.xlabel("Epoch")
    plt.ylabel("Validation Loss")
    plt.legend(fontsize=7, ncol=2)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "all_models_validation_loss_curves_stage1_only.png", dpi=300)
    plt.show()
    plt.close()

    if any("val_auc" in hdf.columns for hdf in all_histories.values()):
        plt.figure(figsize=(16, 8))
        for model_name, history_df in all_histories.items():
            if "val_auc" in history_df.columns:
                plt.plot(
                    history_df["val_auc"].values,
                    label=model_name,
                    linewidth=1.5
                )

        plt.title("Validation AUC Curves - All Completed Models - Stage 1 Only")
        plt.xlabel("Epoch")
        plt.ylabel("Validation AUC")
        plt.legend(fontsize=7, ncol=2)
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(PLOT_DIR / "all_models_validation_auc_curves_stage1_only.png", dpi=300)
        plt.show()
        plt.close()

best_summary = {
    "total_models_requested": len(MODEL_CONFIGS),
    "total_models_completed": int((comparison_df["training_status"] == "completed").sum()),
    "total_models_failed": int((comparison_df["training_status"] == "failed").sum()),
    "training_mode": "stage1_only_frozen_backbone",
    "best_model_name": BEST_MODEL_NAME,
    "best_model_key": BEST_MODEL_KEY,
    "best_validation_accuracy": BEST_VAL_ACCURACY,
    "best_model_path": str(BEST_MODEL_SAVED_PATH),
    "best_weights_path": str(BEST_WEIGHTS_SAVED_PATH),
    "comparison_csv_path": str(comparison_csv_path),
    "all_model_runs_dir": str(ALL_MODELS_DIR)
}

with open(STATS_DIR / "best_model_summary_25_models_stage1_only.json", "w") as f:
    json.dump(best_summary, f, indent=4)

print("\n" + "=" * 100)
print("BEST MODEL SUMMARY - STAGE 1 ONLY")
print("=" * 100)
print(json.dumps(best_summary, indent=4))

print("Comparison CSV saved at:", comparison_csv_path)
print("Best model saved at:", BEST_MODEL_SAVED_PATH)
print("Best weights saved at:", BEST_WEIGHTS_SAVED_PATH)
print("Plots saved at:", PLOT_DIR)
print("Cell 6 completed successfully.")


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    auc,
    matthews_corrcoef,
    cohen_kappa_score
)
from sklearn.preprocessing import label_binarize

import tensorflow as tf
from tensorflow import keras

SEED = globals().get("SEED", 42)
IMG_SIZE = globals().get("IMG_SIZE", 224)
BATCH_SIZE = globals().get("BATCH_SIZE", 16)
NUM_CLASSES = globals().get(
    "NUM_CLASSES",
    len(class_names) if "class_names" in globals() else 6
)

WORK_DIR = Path("/kaggle/working")
PLOT_DIR = WORK_DIR / "plots"
STATS_DIR = WORK_DIR / "stats"
MODEL_DIR = WORK_DIR / "models"
XAI_DIR = WORK_DIR / "xai_outputs"

for d in [PLOT_DIR, STATS_DIR, MODEL_DIR, XAI_DIR]:
    d.mkdir(parents=True, exist_ok=True)

required_vars = [
    "test_gen",
    "X_test",
    "y_test",
    "class_names",
    "MODEL_CONFIGS",
    "build_model_from_config",
    "sanitize_name"
]

for var in required_vars:
    if var not in globals():
        raise NameError(f"{var} is not defined. Run Cells 1 to 6 first.")

idx_to_class = globals().get(
    "idx_to_class",
    {i: c for i, c in enumerate(class_names)}
)

BEST_MODEL_DIR = MODEL_DIR / "best_model_stage1_only"
best_info_path = BEST_MODEL_DIR / "best_model_info.json"

if not best_info_path.exists():
    raise FileNotFoundError(
        f"Best model info not found: {best_info_path}. Run Cell 6 first."
    )

with open(best_info_path, "r") as f:
    best_info = json.load(f)

best_model_name = best_info["best_model_name"]
best_model_key = best_info["best_model_key"]
best_weights_path = Path(best_info["best_weights_saved_path"])
best_model_saved_path = Path(best_info["best_model_saved_path"])

print("Best Model Name:", best_model_name)
print("Best Model Key:", best_model_key)
print("Best Weights Path:", best_weights_path)
print("Best Saved Model Path:", best_model_saved_path)

if not best_weights_path.exists():
    raise FileNotFoundError(f"Best weights not found: {best_weights_path}")

best_config = None

for config in MODEL_CONFIGS:
    if sanitize_name(config["model_name"]) == best_model_key:
        best_config = config
        break

if best_config is None:
    raise ValueError("Best model configuration not found in MODEL_CONFIGS.")

print("Best Model Config:")
print(best_config)

keras.backend.clear_session()
tf.random.set_seed(SEED)

best_model, best_base_model = build_model_from_config(
    config=best_config,
    img_size=IMG_SIZE,
    num_classes=NUM_CLASSES
)

best_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=[
        "accuracy",
        keras.metrics.AUC(name="auc", multi_label=True)
    ]
)

best_model.load_weights(str(best_weights_path))

print("Best model loaded successfully.")

test_loss, test_accuracy, test_keras_auc = best_model.evaluate(
    test_gen,
    verbose=1
)

y_prob = best_model.predict(
    test_gen,
    verbose=1
)

y_pred = np.argmax(y_prob, axis=1)
y_true = y_test.copy()

precision_macro = precision_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)

recall_macro = recall_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)

f1_macro = f1_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)

precision_weighted = precision_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

recall_weighted = recall_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

f1_weighted = f1_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

mcc = matthews_corrcoef(
    y_true,
    y_pred
)

kappa = cohen_kappa_score(
    y_true,
    y_pred
)

try:
    macro_auc_ovr = roc_auc_score(
        y_true,
        y_prob,
        multi_class="ovr",
        average="macro"
    )
except Exception as e:
    print("Macro ROC-AUC failed:", e)
    macro_auc_ovr = np.nan

try:
    weighted_auc_ovr = roc_auc_score(
        y_true,
        y_prob,
        multi_class="ovr",
        average="weighted"
    )
except Exception as e:
    print("Weighted ROC-AUC failed:", e)
    weighted_auc_ovr = np.nan

try:
    macro_auc_ovo = roc_auc_score(
        y_true,
        y_prob,
        multi_class="ovo",
        average="macro"
    )
except Exception as e:
    print("Macro OVO ROC-AUC failed:", e)
    macro_auc_ovo = np.nan

metrics_summary = {
    "best_model_name": best_model_name,
    "best_model_key": best_model_key,
    "training_mode": "stage1_only_frozen_backbone",
    "test_loss": float(test_loss),
    "test_accuracy": float(test_accuracy),
    "keras_auc": float(test_keras_auc),
    "macro_precision": float(precision_macro),
    "macro_recall": float(recall_macro),
    "macro_f1_score": float(f1_macro),
    "weighted_precision": float(precision_weighted),
    "weighted_recall": float(recall_weighted),
    "weighted_f1_score": float(f1_weighted),
    "macro_roc_auc_ovr": float(macro_auc_ovr),
    "weighted_roc_auc_ovr": float(weighted_auc_ovr),
    "macro_roc_auc_ovo": float(macro_auc_ovo),
    "mcc": float(mcc),
    "cohen_kappa": float(kappa),
    "best_weights_path": str(best_weights_path),
    "best_model_saved_path": str(best_model_saved_path)
}

metrics_json_path = STATS_DIR / "best_stage1_model_test_metrics_summary.json"

with open(metrics_json_path, "w") as f:
    json.dump(metrics_summary, f, indent=4)

print(json.dumps(metrics_summary, indent=4))

report = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    digits=4,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()

report_csv_path = STATS_DIR / "best_stage1_model_classification_report.csv"

report_df.to_csv(report_csv_path)

print("Classification Report:")
print(report_df)

pred_df = pd.DataFrame({
    "true_idx": y_true,
    "pred_idx": y_pred,
    "true_label": [idx_to_class[int(i)] for i in y_true],
    "pred_label": [idx_to_class[int(i)] for i in y_pred],
    "confidence": np.max(y_prob, axis=1)
})

for i, class_name in enumerate(class_names):
    pred_df[f"prob_{class_name}"] = y_prob[:, i]

pred_csv_path = STATS_DIR / "best_stage1_model_test_predictions.csv"

pred_df.to_csv(
    pred_csv_path,
    index=False
)

cm = confusion_matrix(
    y_true,
    y_pred
)

cm_df = pd.DataFrame(
    cm,
    index=class_names,
    columns=class_names
)

cm_csv_path = STATS_DIR / "best_stage1_model_confusion_matrix_raw.csv"

cm_df.to_csv(cm_csv_path)

plt.figure(figsize=(10, 9))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

disp.plot(
    xticks_rotation=45,
    values_format="d",
    cmap="Blues"
)

plt.title(f"Confusion Matrix - {best_model_name}")
plt.tight_layout()

cm_plot_path = PLOT_DIR / "best_stage1_model_confusion_matrix_raw.png"

plt.savefig(
    cm_plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

cm_norm = confusion_matrix(
    y_true,
    y_pred,
    normalize="true"
)

cm_norm_df = pd.DataFrame(
    cm_norm,
    index=class_names,
    columns=class_names
)

cm_norm_csv_path = STATS_DIR / "best_stage1_model_confusion_matrix_normalized.csv"

cm_norm_df.to_csv(cm_norm_csv_path)

plt.figure(figsize=(10, 9))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_norm,
    display_labels=class_names
)

disp.plot(
    xticks_rotation=45,
    values_format=".2f",
    cmap="Blues"
)

plt.title(f"Normalized Confusion Matrix - {best_model_name}")
plt.tight_layout()

cm_norm_plot_path = PLOT_DIR / "best_stage1_model_confusion_matrix_normalized.png"

plt.savefig(
    cm_norm_plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

metric_bar = pd.DataFrame({
    "metric": [
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1",
        "Weighted Precision",
        "Weighted Recall",
        "Weighted F1",
        "Macro ROC-AUC OVR",
        "Weighted ROC-AUC OVR",
        "Macro ROC-AUC OVO",
        "MCC",
        "Kappa"
    ],
    "value": [
        test_accuracy,
        precision_macro,
        recall_macro,
        f1_macro,
        precision_weighted,
        recall_weighted,
        f1_weighted,
        macro_auc_ovr,
        weighted_auc_ovr,
        macro_auc_ovo,
        mcc,
        kappa
    ]
})

metric_bar_csv_path = STATS_DIR / "best_stage1_model_metric_comparison.csv"

metric_bar.to_csv(
    metric_bar_csv_path,
    index=False
)

plt.figure(figsize=(14, 6))

plt.bar(
    metric_bar["metric"],
    metric_bar["value"]
)

plt.title(f"Best Stage 1 Model Test Performance - {best_model_name}")
plt.ylabel("Score")
plt.xticks(rotation=45, ha="right")
plt.ylim(-1, 1)
plt.grid(axis="y")
plt.tight_layout()

metric_bar_plot_path = PLOT_DIR / "best_stage1_model_metric_comparison_bar.png"

plt.savefig(
    metric_bar_plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

y_true_bin = label_binarize(
    y_true,
    classes=list(range(NUM_CLASSES))
)

plt.figure(figsize=(10, 8))

roc_rows = []

for i, class_name in enumerate(class_names):
    try:
        fpr, tpr, _ = roc_curve(
            y_true_bin[:, i],
            y_prob[:, i]
        )

        class_auc = auc(
            fpr,
            tpr
        )

        roc_rows.append({
            "class_name": class_name,
            "auc": class_auc
        })

        plt.plot(
            fpr,
            tpr,
            label=f"{class_name} AUC={class_auc:.3f}"
        )

    except Exception as e:
        print("ROC failed for:", class_name, e)

plt.plot([0, 1], [0, 1], linestyle="--")
plt.title(f"ROC Curves - {best_model_name}")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(fontsize=8)
plt.grid(True)
plt.tight_layout()

roc_plot_path = PLOT_DIR / "best_stage1_model_roc_curves.png"

plt.savefig(
    roc_plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

roc_df = pd.DataFrame(roc_rows)

roc_csv_path = STATS_DIR / "best_stage1_model_per_class_roc_auc.csv"

roc_df.to_csv(
    roc_csv_path,
    index=False
)

per_class_metrics = []

for i, class_name in enumerate(class_names):
    true_binary = (y_true == i).astype(int)
    pred_binary = (y_pred == i).astype(int)

    class_precision = precision_score(
        true_binary,
        pred_binary,
        zero_division=0
    )

    class_recall = recall_score(
        true_binary,
        pred_binary,
        zero_division=0
    )

    class_f1 = f1_score(
        true_binary,
        pred_binary,
        zero_division=0
    )

    class_mcc = matthews_corrcoef(
        true_binary,
        pred_binary
    )

    try:
        class_auc_value = roc_auc_score(
            true_binary,
            y_prob[:, i]
        )
    except Exception:
        class_auc_value = np.nan

    per_class_metrics.append({
        "class_name": class_name,
        "precision": class_precision,
        "recall": class_recall,
        "f1_score": class_f1,
        "auc": class_auc_value,
        "mcc_one_vs_rest": class_mcc
    })

per_class_metrics_df = pd.DataFrame(per_class_metrics)

per_class_metrics_csv_path = STATS_DIR / "best_stage1_model_per_class_metrics.csv"

per_class_metrics_df.to_csv(
    per_class_metrics_csv_path,
    index=False
)

print("Per-class metrics:")
print(per_class_metrics_df)

plt.figure(figsize=(12, 6))

x = np.arange(len(class_names))
width = 0.25

plt.bar(
    x - width,
    per_class_metrics_df["precision"],
    width,
    label="Precision"
)

plt.bar(
    x,
    per_class_metrics_df["recall"],
    width,
    label="Recall"
)

plt.bar(
    x + width,
    per_class_metrics_df["f1_score"],
    width,
    label="F1-score"
)

plt.xticks(x, class_names, rotation=35, ha="right")
plt.title(f"Per-Class Precision, Recall, and F1-score - {best_model_name}")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.legend()
plt.grid(axis="y")
plt.tight_layout()

per_class_plot_path = PLOT_DIR / "best_stage1_model_per_class_precision_recall_f1.png"

plt.savefig(
    per_class_plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

correct_indices = np.where(y_pred == y_true)[0]
wrong_indices = np.where(y_pred != y_true)[0]

print("Correct predictions:", len(correct_indices))
print("Wrong predictions:", len(wrong_indices))

def show_prediction_grid(indices, title, save_path, n=12):
    if len(indices) == 0:
        print("No samples for:", title)
        return

    chosen = np.random.choice(
        indices,
        size=min(n, len(indices)),
        replace=False
    )

    rows = 3
    cols = 4

    plt.figure(figsize=(18, 12))

    for j, idx in enumerate(chosen):
        plt.subplot(rows, cols, j + 1)

        plt.imshow(X_test[idx].astype(np.uint8))

        true_label = idx_to_class[int(y_true[idx])]
        pred_label = idx_to_class[int(y_pred[idx])]
        confidence = float(np.max(y_prob[idx]))

        plt.title(
            f"T: {true_label}\nP: {pred_label}\nC: {confidence:.2f}",
            fontsize=8
        )

        plt.axis("off")

    plt.suptitle(title)
    plt.tight_layout()

    plt.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

show_prediction_grid(
    correct_indices,
    f"Correct Predictions - {best_model_name}",
    PLOT_DIR / "best_stage1_model_correct_predictions.png"
)

show_prediction_grid(
    wrong_indices,
    f"Wrong Predictions - {best_model_name}",
    PLOT_DIR / "best_stage1_model_wrong_predictions.png"
)

best_model_results_summary = {
    "metrics_json_path": str(metrics_json_path),
    "classification_report_csv": str(report_csv_path),
    "predictions_csv": str(pred_csv_path),
    "confusion_matrix_raw_csv": str(cm_csv_path),
    "confusion_matrix_normalized_csv": str(cm_norm_csv_path),
    "metric_comparison_csv": str(metric_bar_csv_path),
    "roc_auc_csv": str(roc_csv_path),
    "per_class_metrics_csv": str(per_class_metrics_csv_path),
    "plots": {
        "confusion_matrix_raw": str(cm_plot_path),
        "confusion_matrix_normalized": str(cm_norm_plot_path),
        "metric_bar": str(metric_bar_plot_path),
        "roc_curves": str(roc_plot_path),
        "per_class_prf": str(per_class_plot_path)
    }
}

with open(STATS_DIR / "best_stage1_model_results_file_index.json", "w") as f:
    json.dump(best_model_results_summary, f, indent=4)

xai_model = best_model
test_model = best_model

print("Best model is now available as: best_model, xai_model, test_model")
print("Results saved in:", STATS_DIR)
print("Plots saved in:", PLOT_DIR)
print("Cell 7 completed successfully.")


In [ ]:
# CELL 8
# XAI FOR BEST STAGE-1 MODEL
# Grad-CAM, Integrated Gradients, Saliency Maps, Occlusion Maps

from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

import tensorflow as tf
from tensorflow import keras

SEED = globals().get("SEED", 42)
IMG_SIZE = globals().get("IMG_SIZE", 224)
BATCH_SIZE = globals().get("BATCH_SIZE", 16)
N_XAI = globals().get("N_XAI", 10)

WORK_DIR = Path("/kaggle/working")
PLOT_DIR = WORK_DIR / "plots"
STATS_DIR = WORK_DIR / "stats"
MODEL_DIR = WORK_DIR / "models"
XAI_DIR = WORK_DIR / "xai_outputs"

for d in [PLOT_DIR, STATS_DIR, MODEL_DIR, XAI_DIR]:
    d.mkdir(parents=True, exist_ok=True)

required_vars = [
    "X_test",
    "y_test",
    "class_names",
    "MODEL_CONFIGS",
    "build_model_from_config",
    "sanitize_name"
]

for var in required_vars:
    if var not in globals():
        raise NameError(f"{var} is not defined. Run Cells 1 to 7 first.")

NUM_CLASSES = globals().get("NUM_CLASSES", len(class_names))

idx_to_class = globals().get(
    "idx_to_class",
    {i: c for i, c in enumerate(class_names)}
)

BEST_MODEL_DIR = MODEL_DIR / "best_model_stage1_only"
best_info_path = BEST_MODEL_DIR / "best_model_info.json"

if not best_info_path.exists():
    raise FileNotFoundError(
        f"Best model info file not found: {best_info_path}. Run Cell 6 first."
    )

with open(best_info_path, "r") as f:
    best_info = json.load(f)

best_model_name = best_info["best_model_name"]
best_model_key = best_info["best_model_key"]
best_weights_path = Path(best_info["best_weights_saved_path"])

print("Best model for XAI:", best_model_name)
print("Best model key:", best_model_key)
print("Best weights:", best_weights_path)

if not best_weights_path.exists():
    raise FileNotFoundError(f"Best weights not found: {best_weights_path}")

best_config = None

for config in MODEL_CONFIGS:
    if sanitize_name(config["model_name"]) == best_model_key:
        best_config = config
        break

if best_config is None:
    raise ValueError("Best model config not found in MODEL_CONFIGS.")

print("Best config:")
print(best_config)

# Use model from Cell 7 if available, otherwise rebuild and load weights
if "xai_model" in globals():
    print("Using xai_model already created in Cell 7.")
else:
    print("Loading best Stage-1 model for XAI...")

    keras.backend.clear_session()
    tf.random.set_seed(SEED)

    xai_model, xai_base_model = build_model_from_config(
        config=best_config,
        img_size=IMG_SIZE,
        num_classes=NUM_CLASSES
    )

    xai_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="categorical_crossentropy",
        metrics=[
            "accuracy",
            keras.metrics.AUC(name="auc", multi_label=True)
        ]
    )

    xai_model.load_weights(str(best_weights_path))

    print("Best Stage-1 model loaded successfully for XAI.")

# If Cell 7 predictions exist, reuse them
if "y_prob" not in globals() or "y_pred" not in globals():
    print("Predictions not found. Predicting test set for XAI sample selection...")

    y_prob = xai_model.predict(
        X_test.astype(np.float32),
        batch_size=BATCH_SIZE,
        verbose=1
    )

    y_pred = np.argmax(y_prob, axis=1)

y_true = y_test.copy()

# Select samples: top confident + random
confidences = np.max(y_prob, axis=1)
sorted_idx = np.argsort(-confidences)

selected_indices = list(sorted_idx[:min(5, len(sorted_idx))])

remaining = [
    i for i in range(len(X_test))
    if i not in selected_indices
]

if len(remaining) > 0:
    random_extra = np.random.choice(
        remaining,
        size=min(N_XAI - len(selected_indices), len(remaining)),
        replace=False
    )

    selected_indices += list(random_extra)

selected_indices = selected_indices[:N_XAI]

selected_xai_df = pd.DataFrame({
    "test_index": selected_indices,
    "true_label": [idx_to_class[int(y_true[i])] for i in selected_indices],
    "pred_label": [idx_to_class[int(y_pred[i])] for i in selected_indices],
    "confidence": [float(confidences[i]) for i in selected_indices]
})

selected_xai_csv = STATS_DIR / "best_stage1_xai_selected_test_samples.csv"

selected_xai_df.to_csv(
    selected_xai_csv,
    index=False
)

print("Selected XAI test indices:")
print(selected_xai_df)


def normalize_map(x):
    x = np.array(x).astype(np.float32)
    x = x - np.min(x)
    x = x / (np.max(x) + 1e-8)
    return x


def overlay_heatmap(img, heatmap, alpha=0.40):
    heatmap = cv2.resize(
        heatmap,
        (img.shape[1], img.shape[0])
    )

    heatmap_uint8 = np.uint8(255 * normalize_map(heatmap))

    color_map = cv2.applyColorMap(
        heatmap_uint8,
        cv2.COLORMAP_JET
    )

    color_map = cv2.cvtColor(
        color_map,
        cv2.COLOR_BGR2RGB
    )

    overlay = cv2.addWeighted(
        img.astype(np.uint8),
        1 - alpha,
        color_map,
        alpha,
        0
    )

    return overlay


def save_xai_figure(img, heatmap, overlay, title, save_path):
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(img.astype(np.uint8))
    plt.title("Input Image")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(heatmap, cmap="jet")
    plt.title("Explanation Map")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(overlay.astype(np.uint8))
    plt.title(title)
    plt.axis("off")

    plt.tight_layout()
    plt.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()


def gradcam_heatmap(model, img_array, class_idx, conv_layer_name="xai_conv"):
    if conv_layer_name not in [layer.name for layer in model.layers]:
        conv_layers = [
            layer.name for layer in model.layers
            if isinstance(layer, tf.keras.layers.Conv2D)
        ]

        if len(conv_layers) == 0:
            raise ValueError("No Conv2D layer found for Grad-CAM.")

        conv_layer_name = conv_layers[-1]
        print("xai_conv not found. Using last Conv2D layer:", conv_layer_name)

    grad_model = keras.Model(
        inputs=model.inputs,
        outputs=[
            model.get_layer(conv_layer_name).output,
            model.output
        ]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(
            img_array,
            training=False
        )

        loss = predictions[:, class_idx]

    grads = tape.gradient(
        loss,
        conv_outputs
    )

    pooled_grads = tf.reduce_mean(
        grads,
        axis=(0, 1, 2)
    )

    conv_outputs = conv_outputs[0]

    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)

    return heatmap.numpy()


def integrated_gradients(model, img_batch, class_idx, steps=32):
    img_batch = tf.cast(img_batch, tf.float32)
    baseline = tf.zeros_like(img_batch)

    alphas = tf.linspace(0.0, 1.0, steps + 1)

    interpolated_images = baseline + alphas[:, None, None, None] * (
        img_batch - baseline
    )

    with tf.GradientTape() as tape:
        tape.watch(interpolated_images)

        predictions = model(
            interpolated_images,
            training=False
        )

        target = predictions[:, class_idx]

    gradients = tape.gradient(
        target,
        interpolated_images
    )

    avg_gradients = tf.reduce_mean(
        (gradients[:-1] + gradients[1:]) / 2.0,
        axis=0
    )

    integrated_grads = (
        img_batch[0] - baseline[0]
    ) * avg_gradients

    attribution_map = tf.reduce_sum(
        tf.abs(integrated_grads),
        axis=-1
    )

    return normalize_map(attribution_map.numpy())


def saliency_map(model, img_batch, class_idx):
    img_tensor = tf.cast(img_batch, tf.float32)

    with tf.GradientTape() as tape:
        tape.watch(img_tensor)

        predictions = model(
            img_tensor,
            training=False
        )

        loss = predictions[:, class_idx]

    gradients = tape.gradient(
        loss,
        img_tensor
    )[0]

    saliency = tf.reduce_max(
        tf.abs(gradients),
        axis=-1
    )

    return normalize_map(saliency.numpy())


def occlusion_map(model, img_np, class_idx, patch_size=28, stride=28):
    img = img_np.copy().astype(np.float32)

    base_prob = model.predict(
        img[None, ...],
        verbose=0
    )[0, class_idx]

    occ_map = np.zeros(
        (IMG_SIZE, IMG_SIZE),
        dtype=np.float32
    )

    count_map = np.zeros(
        (IMG_SIZE, IMG_SIZE),
        dtype=np.float32
    )

    mean_pixel = np.mean(
        img,
        axis=(0, 1)
    )

    for y in range(0, IMG_SIZE - patch_size + 1, stride):
        for x in range(0, IMG_SIZE - patch_size + 1, stride):
            occluded_img = img.copy()

            occluded_img[
                y:y + patch_size,
                x:x + patch_size,
                :
            ] = mean_pixel

            occluded_prob = model.predict(
                occluded_img[None, ...],
                verbose=0
            )[0, class_idx]

            probability_drop = base_prob - occluded_prob

            occ_map[
                y:y + patch_size,
                x:x + patch_size
            ] += probability_drop

            count_map[
                y:y + patch_size,
                x:x + patch_size
            ] += 1

    occ_map = occ_map / (count_map + 1e-8)

    return normalize_map(occ_map)


gradcam_records = []
ig_records = []
saliency_records = []
occlusion_records = []

print("Generating XAI maps for best Stage-1 model...")

for rank, idx in enumerate(selected_indices):
    img = X_test[idx].astype(np.float32)
    img_batch = img[None, ...]

    probs = xai_model.predict(
        img_batch,
        verbose=0
    )[0]

    pred_class = int(np.argmax(probs))
    true_class = int(y_true[idx])
    confidence = float(probs[pred_class])

    title = (
        f"{best_model_name}\n"
        f"T: {idx_to_class[true_class]}\n"
        f"P: {idx_to_class[pred_class]} | C: {confidence:.2f}"
    )

    gradcam = gradcam_heatmap(
        model=xai_model,
        img_array=img_batch,
        class_idx=pred_class,
        conv_layer_name="xai_conv"
    )

    gradcam_overlay = overlay_heatmap(
        img=img,
        heatmap=gradcam
    )

    gradcam_path = XAI_DIR / f"{rank:02d}_best_stage1_gradcam_testidx_{idx}.png"

    save_xai_figure(
        img=img,
        heatmap=gradcam,
        overlay=gradcam_overlay,
        title=title,
        save_path=gradcam_path
    )

    gradcam_records.append({
        "rank": rank,
        "test_index": int(idx),
        "true_label": idx_to_class[true_class],
        "pred_label": idx_to_class[pred_class],
        "confidence": confidence,
        "best_model_name": best_model_name,
        "xai_method": "Grad-CAM",
        "file_path": str(gradcam_path)
    })

    ig = integrated_gradients(
        model=xai_model,
        img_batch=img_batch,
        class_idx=pred_class,
        steps=32
    )

    ig_overlay = overlay_heatmap(
        img=img,
        heatmap=ig
    )

    ig_path = XAI_DIR / f"{rank:02d}_best_stage1_integrated_gradients_testidx_{idx}.png"

    save_xai_figure(
        img=img,
        heatmap=ig,
        overlay=ig_overlay,
        title=title,
        save_path=ig_path
    )

    ig_records.append({
        "rank": rank,
        "test_index": int(idx),
        "true_label": idx_to_class[true_class],
        "pred_label": idx_to_class[pred_class],
        "confidence": confidence,
        "best_model_name": best_model_name,
        "xai_method": "Integrated Gradients",
        "file_path": str(ig_path)
    })

    saliency = saliency_map(
        model=xai_model,
        img_batch=img_batch,
        class_idx=pred_class
    )

    saliency_overlay = overlay_heatmap(
        img=img,
        heatmap=saliency
    )

    saliency_path = XAI_DIR / f"{rank:02d}_best_stage1_saliency_testidx_{idx}.png"

    save_xai_figure(
        img=img,
        heatmap=saliency,
        overlay=saliency_overlay,
        title=title,
        save_path=saliency_path
    )

    saliency_records.append({
        "rank": rank,
        "test_index": int(idx),
        "true_label": idx_to_class[true_class],
        "pred_label": idx_to_class[pred_class],
        "confidence": confidence,
        "best_model_name": best_model_name,
        "xai_method": "Saliency Map",
        "file_path": str(saliency_path)
    })

    occlusion = occlusion_map(
        model=xai_model,
        img_np=img,
        class_idx=pred_class,
        patch_size=28,
        stride=28
    )

    occlusion_overlay = overlay_heatmap(
        img=img,
        heatmap=occlusion
    )

    occlusion_path = XAI_DIR / f"{rank:02d}_best_stage1_occlusion_testidx_{idx}.png"

    save_xai_figure(
        img=img,
        heatmap=occlusion,
        overlay=occlusion_overlay,
        title=title,
        save_path=occlusion_path
    )

    occlusion_records.append({
        "rank": rank,
        "test_index": int(idx),
        "true_label": idx_to_class[true_class],
        "pred_label": idx_to_class[pred_class],
        "confidence": confidence,
        "best_model_name": best_model_name,
        "xai_method": "Occlusion Map",
        "file_path": str(occlusion_path)
    })

xai_records_df = pd.DataFrame(
    gradcam_records + ig_records + saliency_records + occlusion_records
)

xai_index_path = STATS_DIR / "best_stage1_cell8_xai_outputs_index.csv"

xai_records_df.to_csv(
    xai_index_path,
    index=False
)

print("Best Stage-1 XAI outputs saved at:", XAI_DIR)
print("XAI index saved at:", xai_index_path)
print("Selected samples saved at:", selected_xai_csv)
print("Cell 8 completed successfully.")

In [ ]:
from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from lime import lime_image
except Exception:
    !pip -q install lime
    from lime import lime_image

try:
    from skimage.segmentation import mark_boundaries
except Exception:
    !pip -q install scikit-image
    from skimage.segmentation import mark_boundaries

SEED = globals().get("SEED", 42)
IMG_SIZE = globals().get("IMG_SIZE", 224)
BATCH_SIZE = globals().get("BATCH_SIZE", 16)
N_XAI = globals().get("N_XAI", 10)

WORK_DIR = Path("/kaggle/working")
PLOT_DIR = WORK_DIR / "plots"
STATS_DIR = WORK_DIR / "stats"
MODEL_DIR = WORK_DIR / "models"
XAI_DIR = WORK_DIR / "xai_outputs"

for d in [PLOT_DIR, STATS_DIR, MODEL_DIR, XAI_DIR]:
    d.mkdir(parents=True, exist_ok=True)

required_vars = [
    "xai_model",
    "X_test",
    "y_test",
    "y_pred",
    "selected_indices",
    "class_names"
]

for var in required_vars:
    if var not in globals():
        raise NameError(f"{var} is not defined. Run Cells 1 to 9 first.")

NUM_CLASSES = globals().get("NUM_CLASSES", len(class_names))

idx_to_class = globals().get(
    "idx_to_class",
    {i: c for i, c in enumerate(class_names)}
)

np.random.seed(SEED)

print("Starting Cell 10: LIME + final output export")


def lime_predict_fn(images):
    images = np.array(images).astype(np.float32)
    return xai_model.predict(
        images,
        batch_size=BATCH_SIZE,
        verbose=0
    )


lime_records = []

try:
    lime_explainer = lime_image.LimeImageExplainer(
        random_state=SEED
    )

    print("Generating LIME explanations...")

    for rank, test_idx in enumerate(selected_indices):
        img = X_test[test_idx].astype(np.uint8)
        pred_class = int(y_pred[test_idx])
        true_class = int(y_test[test_idx])

        explanation = lime_explainer.explain_instance(
            image=img,
            classifier_fn=lime_predict_fn,
            top_labels=NUM_CLASSES,
            hide_color=0,
            num_samples=600
        )

        temp, mask = explanation.get_image_and_mask(
            label=pred_class,
            positive_only=True,
            num_features=8,
            hide_rest=False
        )

        lime_img = mark_boundaries(
            temp / 255.0,
            mask
        )

        lime_path = XAI_DIR / f"{rank:02d}_lime_testidx_{test_idx}.png"

        plt.figure(figsize=(8, 6))
        plt.imshow(lime_img)
        plt.title(
            f"LIME\n"
            f"T: {idx_to_class[true_class]}\n"
            f"P: {idx_to_class[pred_class]}"
        )
        plt.axis("off")
        plt.tight_layout()
        plt.savefig(lime_path, dpi=300, bbox_inches="tight")
        plt.show()
        plt.close()

        lime_records.append({
            "rank": rank,
            "test_index": int(test_idx),
            "true_label": idx_to_class[true_class],
            "pred_label": idx_to_class[pred_class],
            "xai_method": "LIME",
            "file_path": str(lime_path)
        })

except Exception as e:
    print("LIME failed:", e)

lime_records_df = pd.DataFrame(lime_records)

lime_records_df.to_csv(
    STATS_DIR / "cell10_lime_outputs_index.csv",
    index=False
)

metrics_path = STATS_DIR / "test_metrics_summary.json"

if metrics_path.exists():
    with open(metrics_path, "r") as f:
        metrics_summary = json.load(f)
else:
    metrics_summary = {}

xai_files = sorted([str(p) for p in XAI_DIR.glob("*.png")])
plot_files = sorted([str(p) for p in PLOT_DIR.glob("*.png")])
stats_files = sorted([str(p) for p in STATS_DIR.glob("*")])
model_files = sorted([str(p) for p in MODEL_DIR.glob("*")])

final_summary = {
    "experiment_name": "Ultra-wide Fundus Tumor Diagnosis with CLAHE, SMOTE, EfficientNetB0, and XAI",
    "dataset_path": "/kaggle/input/datasets/nikitamanaenkov/ultra-wide-fundus-images-for-tumor-diagnosis/data",
    "classes": class_names,
    "num_classes": NUM_CLASSES,
    "image_size": IMG_SIZE,
    "split_strategy": "Stratified 70-15-15",
    "preprocessing": {
        "clahe_clip_limit": globals().get("CLAHE_CLIP_LIMIT", 1.0),
        "gaussian_blur": globals().get("USE_GAUSSIAN_BLUR", True),
        "black_border_crop": True,
        "ram_loading": True
    },
    "balancing": {
        "method": "SMOTE",
        "applied_on": "Train set only",
        "validation_test_status": "Unchanged, no SMOTE"
    },
    "model": {
        "architecture": "EfficientNetB0",
        "custom_layers": [
            "xai_conv",
            "BatchNormalization",
            "GlobalAveragePooling2D",
            "Dropout",
            "Dense Softmax"
        ],
        "best_weights": str(MODEL_DIR / "best_model_weights.weights.h5"),
        "final_weights": str(MODEL_DIR / "final_model_weights.weights.h5"),
        "final_model": str(MODEL_DIR / "final_full_model.keras")
    },
    "metrics": metrics_summary,
    "xai_methods_completed": [
        "Grad-CAM",
        "Integrated Gradients",
        "Saliency Maps",
        "Occlusion Maps",
        "Tree SHAP on CNN embeddings using RandomForest surrogate",
        "Kernel SHAP on CNN embeddings",
        "Gradient SHAP",
        "LIME"
    ],
    "output_directories": {
        "plots": str(PLOT_DIR),
        "stats": str(STATS_DIR),
        "models": str(MODEL_DIR),
        "xai_outputs": str(XAI_DIR)
    },
    "saved_file_counts": {
        "plots_png": len(plot_files),
        "xai_png": len(xai_files),
        "stats_files": len(stats_files),
        "model_files": len(model_files)
    },
    "important_note": "This notebook is for research only and must not be used for clinical diagnosis without expert medical validation."
}

summary_path = WORK_DIR / "final_experiment_summary.json"

with open(summary_path, "w") as f:
    json.dump(final_summary, f, indent=4)

print(json.dumps(final_summary, indent=4))

all_xai_index_files = [
    STATS_DIR / "cell8_xai_outputs_index.csv",
    STATS_DIR / "cell9_shap_outputs_index.csv",
    STATS_DIR / "cell10_lime_outputs_index.csv"
]

existing_index_files = [
    file for file in all_xai_index_files
    if file.exists()
]

if len(existing_index_files) > 0:
    combined_xai_index = pd.concat(
        [pd.read_csv(file) for file in existing_index_files],
        axis=0,
        ignore_index=True
    )

    combined_xai_index.to_csv(
        STATS_DIR / "combined_xai_outputs_index.csv",
        index=False
    )

    print("Combined XAI index saved at:", STATS_DIR / "combined_xai_outputs_index.csv")

zip_path = WORK_DIR / "uwf_fundus_tumor_complete_outputs.zip"

!zip -r /kaggle/working/uwf_fundus_tumor_complete_outputs.zip \
/kaggle/working/plots \
/kaggle/working/stats \
/kaggle/working/models \
/kaggle/working/xai_outputs \
/kaggle/working/final_experiment_summary.json

print("Final ZIP saved at:", zip_path)
print("Cell 10 completed successfully.")

In [ ]:
# SHAP IMAGE PLOT LIKE YOUR SAMPLE

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt

try:
    import shap
except Exception:
    !pip -q install shap
    import shap

SEED = globals().get("SEED", 42)
IMG_SIZE = globals().get("IMG_SIZE", 224)
BATCH_SIZE = globals().get("BATCH_SIZE", 16)

WORK_DIR = Path("/kaggle/working")
XAI_DIR = WORK_DIR / "xai_outputs"
STATS_DIR = WORK_DIR / "stats"

XAI_DIR.mkdir(parents=True, exist_ok=True)
STATS_DIR.mkdir(parents=True, exist_ok=True)

required_vars = [
    "xai_model",
    "X_train_smote",
    "X_test",
    "y_test",
    "class_names"
]

for var in required_vars:
    if var not in globals():
        raise NameError(f"{var} is not defined. Run previous cells first.")

NUM_CLASSES = len(class_names)
np.random.seed(SEED)

# Take 5 test images like your screenshot
n_shap_images = 5

if "selected_indices" in globals():
    shap_indices = selected_indices[:n_shap_images]
else:
    shap_indices = np.random.choice(
        len(X_test),
        size=n_shap_images,
        replace=False
    ).tolist()

xai_images = X_test[shap_indices].astype(np.float32)

# Background images for SHAP
background_size = min(25, len(X_train_smote))

background_indices = np.random.choice(
    len(X_train_smote),
    size=background_size,
    replace=False
)

background_images = X_train_smote[background_indices].astype(np.float32)

print("SHAP test images:", xai_images.shape)
print("SHAP background images:", background_images.shape)

# Gradient SHAP / Deep-like SHAP for TensorFlow CNN
explainer = shap.GradientExplainer(
    xai_model,
    background_images
)

shap_values_raw = explainer.shap_values(xai_images)

# Convert output into list format required by shap.image_plot
# Needed because different SHAP versions return different shapes.
if isinstance(shap_values_raw, list):
    shap_values_for_plot = shap_values_raw
else:
    shap_array = np.array(shap_values_raw)

    if shap_array.ndim == 5:
        # Shape usually: samples, H, W, C, classes
        shap_values_for_plot = [
            shap_array[:, :, :, :, class_idx]
            for class_idx in range(NUM_CLASSES)
        ]
    elif shap_array.ndim == 4:
        # Single output fallback
        shap_values_for_plot = [shap_array]
    else:
        raise ValueError(f"Unexpected SHAP output shape: {shap_array.shape}")

# Make class labels short for plot
short_class_names = []
for c in class_names:
    short = c.replace("Choroidal Hemangioma", "CH")
    short = short.replace("Choroidal Osteoma", "CO")
    short = short.replace("Retinal Capillary Hemangioma", "RCH")
    short = short.replace("Retinoblastoma", "RB")
    short = short.replace("Uveal Melanoma", "UM")
    short = short.replace("Normal", "Normal")
    short_class_names.append(short)

# This produces the same style:
# original image left side + SHAP red/blue maps for classes
plt.figure(figsize=(18, 12))

shap.image_plot(
    shap_values_for_plot,
    xai_images.astype(np.uint8),
    labels=short_class_names,
    show=False
)

save_path = XAI_DIR / "shap_image_plot_like_reference.png"

plt.tight_layout()
plt.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved SHAP image plot at:", save_path)
print("SHAP image plot completed successfully.")

In [ ]:
# EXTRA SHAP CELL: IMAGE SHAP + DECISION + BAR + HEATMAP + FORCE PLOTS

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import shap
except Exception:
    !pip -q install shap
    import shap

import tensorflow as tf
from tensorflow import keras

SEED = globals().get("SEED", 42)
IMG_SIZE = globals().get("IMG_SIZE", 224)
BATCH_SIZE = globals().get("BATCH_SIZE", 16)

WORK_DIR = Path("/kaggle/working")
XAI_DIR = WORK_DIR / "xai_outputs"
STATS_DIR = WORK_DIR / "stats"

XAI_DIR.mkdir(parents=True, exist_ok=True)
STATS_DIR.mkdir(parents=True, exist_ok=True)

required_vars = [
    "xai_model",
    "X_train_smote",
    "X_test",
    "y_test",
    "class_names"
]

for var in required_vars:
    if var not in globals():
        raise NameError(f"{var} is not defined. Run previous cells first.")

NUM_CLASSES = len(class_names)

np.random.seed(SEED)
tf.random.set_seed(SEED)

if "selected_indices" in globals():
    shap_indices = selected_indices[:10]
else:
    shap_indices = np.random.choice(
        len(X_test),
        size=min(10, len(X_test)),
        replace=False
    ).tolist()

print("SHAP selected test indices:", shap_indices)

# ============================================================
# PART A: SHAP IMAGE PLOT LIKE FUNDUS IMAGE + RED/BLUE MAPS
# ============================================================

n_image_plot = min(5, len(shap_indices))

xai_image_indices = shap_indices[:n_image_plot]
xai_images = X_test[xai_image_indices].astype(np.float32)

background_size = min(25, len(X_train_smote))

background_indices = np.random.choice(
    len(X_train_smote),
    size=background_size,
    replace=False
)

background_images = X_train_smote[background_indices].astype(np.float32)

print("Image SHAP input shape:", xai_images.shape)
print("Image SHAP background shape:", background_images.shape)

image_explainer = shap.GradientExplainer(
    xai_model,
    background_images
)

image_shap_values_raw = image_explainer.shap_values(
    xai_images
)

if isinstance(image_shap_values_raw, list):
    image_shap_values_for_plot = image_shap_values_raw
else:
    image_shap_array = np.array(image_shap_values_raw)

    if image_shap_array.ndim == 5:
        image_shap_values_for_plot = [
            image_shap_array[:, :, :, :, class_idx]
            for class_idx in range(NUM_CLASSES)
        ]
    elif image_shap_array.ndim == 4:
        image_shap_values_for_plot = [image_shap_array]
    else:
        raise ValueError(f"Unexpected image SHAP shape: {image_shap_array.shape}")

short_class_names = []

for c in class_names:
    short = c
    short = short.replace("Choroidal Hemangioma", "CH")
    short = short.replace("Choroidal Osteoma", "CO")
    short = short.replace("Retinal Capillary Hemangioma", "RCH")
    short = short.replace("Retinoblastoma", "RB")
    short = short.replace("Uveal Melanoma", "UM")
    short_class_names.append(short)

plt.figure(figsize=(20, 12))

shap.image_plot(
    image_shap_values_for_plot,
    xai_images.astype(np.uint8),
    labels=short_class_names,
    show=False
)

image_plot_path = XAI_DIR / "shap_image_plot_fundus_red_blue.png"

plt.tight_layout()
plt.savefig(
    image_plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

print("Saved image SHAP plot:", image_plot_path)

# ============================================================
# PART B: SHAP DECISION + BAR + HEATMAP + FORCE LIKE YOUR SAMPLE
# Uses CNN embedding-level Kernel SHAP
# ============================================================

embedding_model = keras.Model(
    inputs=xai_model.input,
    outputs=xai_model.get_layer("embedding").output
)

max_background_embed = min(100, len(X_train_smote))

background_embed_indices = np.random.choice(
    len(X_train_smote),
    size=max_background_embed,
    replace=False
)

X_background_images = X_train_smote[background_embed_indices].astype(np.float32)
X_explain_images = X_test[shap_indices].astype(np.float32)

print("Extracting background embeddings...")

X_background_embed = embedding_model.predict(
    X_background_images,
    batch_size=BATCH_SIZE,
    verbose=1
)

print("Extracting explanation embeddings...")

X_explain_embed = embedding_model.predict(
    X_explain_images,
    batch_size=BATCH_SIZE,
    verbose=1
)

W, b = xai_model.get_layer("predictions").get_weights()

def embedding_softmax_predict(z):
    z = np.array(z).astype(np.float32)

    logits = np.dot(z, W) + b

    exp_logits = np.exp(
        logits - np.max(logits, axis=1, keepdims=True)
    )

    probs = exp_logits / np.sum(
        exp_logits,
        axis=1,
        keepdims=True
    )

    return probs

print("Running Kernel SHAP on CNN embeddings...")

kernel_explainer = shap.KernelExplainer(
    embedding_softmax_predict,
    X_background_embed
)

kernel_shap_values = kernel_explainer.shap_values(
    X_explain_embed,
    nsamples=150
)

probs = embedding_softmax_predict(X_explain_embed)
pred_classes = np.argmax(probs, axis=1)

sample_id = 0
target_class = int(pred_classes[sample_id])
target_class_name = class_names[target_class]

print("Target class:", target_class_name)

if isinstance(kernel_shap_values, list):
    shap_values_class = np.array(kernel_shap_values[target_class])
    expected_value_class = kernel_explainer.expected_value[target_class]
else:
    shap_array = np.array(kernel_shap_values)

    if shap_array.ndim == 3:
        shap_values_class = shap_array[:, :, target_class]
        expected_value_class = kernel_explainer.expected_value[target_class]
    elif shap_array.ndim == 2:
        shap_values_class = shap_array
        expected_value_class = kernel_explainer.expected_value
    else:
        raise ValueError(f"Unexpected Kernel SHAP shape: {shap_array.shape}")

feature_names = [
    f"embedding_{i}"
    for i in range(X_explain_embed.shape[1])
]

np.save(
    STATS_DIR / "kernel_shap_values_target_class.npy",
    shap_values_class
)

pd.DataFrame(
    X_explain_embed,
    columns=feature_names
).to_csv(
    STATS_DIR / "kernel_shap_embedding_features.csv",
    index=False
)

pd.DataFrame(
    shap_values_class,
    columns=feature_names
).to_csv(
    STATS_DIR / "kernel_shap_values_target_class.csv",
    index=False
)

# --------------------------
# SHAP decision plot
# --------------------------

plt.figure(figsize=(18, 8))

shap.decision_plot(
    base_value=expected_value_class,
    shap_values=shap_values_class,
    features=X_explain_embed,
    feature_names=feature_names,
    show=False,
    ignore_warnings=True
)

plt.title(f"SHAP Decision Plot - {target_class_name}")
plt.tight_layout()

decision_path = XAI_DIR / "shap_decision_plot_embedding.png"

plt.savefig(
    decision_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

print("Saved decision plot:", decision_path)

# --------------------------
# SHAP bar plot
# --------------------------

sample_shap = shap_values_class[sample_id]
sample_features = X_explain_embed[sample_id]

explanation_single = shap.Explanation(
    values=sample_shap,
    base_values=expected_value_class,
    data=sample_features,
    feature_names=feature_names
)

plt.figure(figsize=(10, 8))

shap.plots.bar(
    explanation_single,
    max_display=20,
    show=False
)

plt.title(f"SHAP Bar Plot - Sample {shap_indices[sample_id]} - {target_class_name}")
plt.tight_layout()

bar_path = XAI_DIR / "shap_bar_plot_single_sample.png"

plt.savefig(
    bar_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

print("Saved bar plot:", bar_path)

# --------------------------
# SHAP heatmap
# --------------------------

explanation_multi = shap.Explanation(
    values=shap_values_class,
    base_values=np.repeat(
        expected_value_class,
        shap_values_class.shape[0]
    ),
    data=X_explain_embed,
    feature_names=feature_names
)

plt.figure(figsize=(16, 8))

shap.plots.heatmap(
    explanation_multi,
    max_display=30,
    show=False
)

plt.title(f"SHAP Heatmap - {target_class_name}")
plt.tight_layout()

heatmap_path = XAI_DIR / "shap_heatmap_embedding.png"

plt.savefig(
    heatmap_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

print("Saved heatmap:", heatmap_path)

# --------------------------
# SHAP force plot PNG
# --------------------------

plt.figure(figsize=(18, 4))

shap.force_plot(
    expected_value_class,
    sample_shap,
    sample_features,
    feature_names=feature_names,
    matplotlib=True,
    show=False
)

plt.title(f"SHAP Force Plot - Sample {shap_indices[sample_id]} - {target_class_name}")
plt.tight_layout()

force_png_path = XAI_DIR / "shap_force_plot_single_sample.png"

plt.savefig(
    force_png_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

print("Saved force PNG:", force_png_path)

# --------------------------
# SHAP force plot HTML
# --------------------------

force_html = shap.force_plot(
    expected_value_class,
    sample_shap,
    sample_features,
    feature_names=feature_names
)

force_html_path = XAI_DIR / "shap_force_plot_single_sample.html"

shap.save_html(
    str(force_html_path),
    force_html
)

print("Saved force HTML:", force_html_path)

# --------------------------
# SHAP waterfall plot
# --------------------------

plt.figure(figsize=(10, 8))

shap.plots.waterfall(
    explanation_single,
    max_display=20,
    show=False
)

plt.title(f"SHAP Waterfall Plot - Sample {shap_indices[sample_id]} - {target_class_name}")
plt.tight_layout()

waterfall_path = XAI_DIR / "shap_waterfall_plot_single_sample.png"

plt.savefig(
    waterfall_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

print("Saved waterfall plot:", waterfall_path)

# --------------------------
# Save index
# --------------------------

shap_plot_index = pd.DataFrame({
    "plot_name": [
        "image_plot_red_blue",
        "decision_plot",
        "bar_plot_single_sample",
        "heatmap",
        "force_plot_png",
        "force_plot_html",
        "waterfall_plot"
    ],
    "file_path": [
        str(image_plot_path),
        str(decision_path),
        str(bar_path),
        str(heatmap_path),
        str(force_png_path),
        str(force_html_path),
        str(waterfall_path)
    ],
    "target_class": [
        "multi_class_image_plot",
        target_class_name,
        target_class_name,
        target_class_name,
        target_class_name,
        target_class_name,
        target_class_name
    ],
    "sample_test_index": [
        ",".join(map(str, xai_image_indices)),
        shap_indices[sample_id],
        shap_indices[sample_id],
        ",".join(map(str, shap_indices)),
        shap_indices[sample_id],
        shap_indices[sample_id],
        shap_indices[sample_id]
    ]
})

shap_plot_index.to_csv(
    STATS_DIR / "shap_all_reference_style_plot_index.csv",
    index=False
)

print("All SHAP reference-style plots completed.")
print(shap_plot_index)

In [ ]:
import os
import shutil
from pathlib import Path
from IPython.display import FileLink, display

# Go to Kaggle working directory
os.chdir("/kaggle/working")

# Exact model path
src = Path("/kaggle/working/models/best_model/best_overall_model.keras")

print("Current directory:", os.getcwd())
print("Looking for model at:", src)

# If exact file not found, search for model file
if not src.exists():
    print("Exact model file not found. Searching in /kaggle/working...")
    
    found_files = list(Path("/kaggle/working").rglob("best_overall_model.keras"))
    
    if found_files:
        src = found_files[0]
        print("Found model at:", src)
    else:
        print("Model not found.")
        print("\nAvailable files inside /kaggle/working/models:")
        
        models_dir = Path("/kaggle/working/models")
        if models_dir.exists():
            for f in models_dir.rglob("*"):
                print(f)
        else:
            print("models folder does not exist.")
        
        raise FileNotFoundError("best_overall_model.keras not found.")

# Zip output path
zip_name = "best_overall_model.zip"
zip_path = Path("/kaggle/working") / zip_name

# Remove old zip if exists
if zip_path.exists():
    zip_path.unlink()

# If .keras is a file
if src.is_file():
    temp_folder = Path("/kaggle/working/temp_model_zip")
    
    if temp_folder.exists():
        shutil.rmtree(temp_folder)
    
    temp_folder.mkdir(parents=True, exist_ok=True)
    
    # Copy model file using only file name
    shutil.copy2(src, temp_folder / src.name)
    
    shutil.make_archive(
        base_name="/kaggle/working/best_overall_model",
        format="zip",
        root_dir=temp_folder
    )
    
    shutil.rmtree(temp_folder)

# If .keras is a folder
elif src.is_dir():
    shutil.make_archive(
        base_name="/kaggle/working/best_overall_model",
        format="zip",
        root_dir=src.parent,
        base_dir=src.name
    )

print("Zip created successfully:", zip_path)
print("Zip size MB:", round(zip_path.stat().st_size / (1024 * 1024), 2))

# Kaggle download link
display(FileLink(zip_name))